# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.scores.intratext_coherence_score import ComputationMethod
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20NG.csv',
 '20NG__internals',
 'Brown',
 'Brown_BOW.csv',
 'Brown_NOOW.csv',
 'MKB10.csv',
 'MKB10__internals',
 'RTL_Wiki.csv',
 'RTL_Wiki_person.csv',
 'RTL_Wiki_person__internals',
 'Reuters',
 'Reuters_BOW.csv',
 'Reuters_NOOW.csv',
 'WikiRef-220',
 '__init__.py',
 '__pycache__',
 'api.py',
 'postnauka.csv',
 'postnauka__internals',
 'ruwiki_good.txt',
 'ruwiki_good__internals',
 'wiki_ref220_bow.csv',
 'wiki_ref220_natural_order.csv']

In [6]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    TWENTY_NEWSGROUPS = '20_Newsgroups'
    GOOD_RU_WIKI = 'Good_RU_Wiki'
    RTL_WIKI_PERSON = 'RTL_Wiki_Person'

In [7]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'postnauka.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    DatasetName.TWENTY_NEWSGROUPS: os.path.join(
        DATA_FOLDER_PATH, '20NG.csv'
    ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
    DatasetName.GOOD_RU_WIKI: os.path.join(
        DATA_FOLDER_PATH, 'ruwiki_good.txt'
    ),
    DatasetName.RTL_WIKI_PERSON: os.path.join(
        DATA_FOLDER_PATH, 'RTL_Wiki_person.csv'
    ),
}

In [8]:
DATASET_NAME = DatasetName.RTL_WIKI_PERSON  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [9]:
! head -n 2 $DATASET_FILE_PATH

,id,raw_text,vw_text
0,İsmet_İnönü,"Mustafa İsmet İnönü (September 24 1884 – December 25, 1973) was a Turkish Army General, TSK Genel Kurmay Baskanlari  Prime Minister and the second President of the Republic of Turkey. He is widely referred to as ""Milli Şef"" (National Chief), a title he bestowed upon himself  when he was elected as the President of Turkey in 1938.  Family and early life He was born in İzmir to a family originally from Malatya with mixed Turkish-Kurdish heritage. The Young Turks – Children of the Borderlands? - Erik Jan Zürcher (Universiteit Leiden)  Ismet Inonu: The Making of a Turkish Statesman - Metin Heper / Brill Academic Publishers  His father was Hacı Reşid Bey, a member of the Ottoman bureaucracy, an examining magistrate born in Malatya, and his mother was Cevriye Hanım, daughter of Russo-Turkish War refugees from Bulgaria. Due to his father's assignments, the family moved from one city to another. Thus, İsmet İnönü completed his primary education in Sivas.  

In [10]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join('.', dataset_name.value + '__internals')

In [11]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [12]:
DATASET_INTERNALS_FOLDER_PATH

'./RTL_Wiki_Person__internals'

In [13]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 809 ms, sys: 120 ms, total: 930 ms
Wall time: 885 ms


Looking what is inside dataset's folder

In [14]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'result2', 'batches', 'dict.dict', 'result']

Creating batches

In [15]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="./RTL_Wiki_Person__internals/batches", num_batches=2)

In [16]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'result2', 'batches', 'dict.dict', 'result']

In [17]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 1201


Let's look at some text samples

In [18]:
DATASET._data.head()

,Unnamed: 0,id,raw_text,vw_text
id,,,,
İsmet_İnönü,0,İsmet_İnönü,Mustafa İsmet İnönü (September 24 1884 – Decem...,İsmet_İnönü |@lemmatized mustafa:2 smet:7 nönü...
Clara_Petacci,1,Clara_Petacci,Clara Petacci (Claretta Petacci) (28 February ...,Clara_Petacci |@lemmatized clara:5 petacci:15 ...
Jack_Ruby,2,Jack_Ruby,"Jacob Rubenstein (March 25, 1911 – January 3, ...",Jack_Ruby |@lemmatized jacob:2 rubenstein:5 ma...
Knud_Rasmussen,3,Knud_Rasmussen,"Knud Johan Victor Rasmussen (June 7, 1879–Dece...",Knud_Rasmussen |@lemmatized knud:15 johan:3 vi...
Gerald_Schroeder,4,Gerald_Schroeder,"Gerald L. Schroeder is a scientist, author, an...",Gerald_Schroeder |@lemmatized gerald:4 l:1 sch...


In [19]:
DATASET.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [20]:
MAIN_MODALITY = '@lemmatized'

In [21]:
DATASET.get_dictionary()

artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=124241)

In [22]:
dictionary = DATASET.get_dictionary()

In [23]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=124241)


In [24]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739)

In [25]:
DATASET._cached_dict = dictionary

In [26]:
DATASET.get_dictionary()

artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739)

In [27]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [28]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [29]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 6.59 s, sys: 232 ms, total: 6.82 s
Wall time: 6.75 s


In [30]:
co_occurences.shape

(37739, 37739)

In [31]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [32]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [33]:
ONE_MODEL_NUM_TOPICS = 50
NUM_TOP_WORDS = 20

In [34]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [35]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [36]:
NUM_ITERATIONS = 20

In [37]:
seed = 0

In [38]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [39]:
DATASET_INTERNALS_FOLDER_PATH

'./RTL_Wiki_Person__internals'

In [40]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result_50'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [41]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls -alh $DATASET_INTERNALS_FOLDER_PATH

./RTL_Wiki_Person__internals
total 17M
drwxrwxr-x  5 alekseev_v mil_lab 4,0K мар 28 02:01 .
drwxrwxr-x 11 alekseev_v mil_lab 4,0K мар 27 23:57 ..
drwxrwxr-x  2 alekseev_v mil_lab 4,0K мар 25 15:05 batches
-rw-rw-r--  1 alekseev_v mil_lab 4,5M мар 25 15:05 dict.dict
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 25 15:06 result
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 25 15:42 result2
-rw-rw-r--  1 alekseev_v mil_lab  12M мар 25 15:05 vw.txt


In [42]:
SEARCH_RESULTS_FOLDER_PATH

'./RTL_Wiki_Person__internals/result_50'

In [43]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access './RTL_Wiki_Person__internals/result_50': No such file or directory


In [44]:
BANK_FOLDER_PATH

'./RTL_Wiki_Person__internals/result_50/bank__0'

In [45]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [46]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [47]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [48]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [49]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls $DATASET_INTERNALS_FOLDER_PATH

./RTL_Wiki_Person__internals
batches  dict.dict  result  result2  result_50	vw.txt


In [50]:
optimizer._save_file_path

'./RTL_Wiki_Person__internals/result_50/search_result__0.json'

In [51]:
optimizer._topic_bank._path

'./RTL_Wiki_Person__internals/result_50/bank__0'

Fulfilling the search (get ready for a really long process!):

In [52]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.86it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 23878.166015625, 'coherence_20': 1.229698071524603, 'diversity_euclidean': 0.0582555565105075, 'diversity_jensenshannon': 0.6403552575649625, 'diversity_hellinger': 0.74680146288752, 'diversity_cosine': 0.8519951977197273, 'perplexity': 23878.166015625, 'ppl_fair': 23878.166015625, 'ppl_cheatty': 6565.55517578125}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.56it/s]
Creating first level with 5 topics. Dictionary: artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739).
Copying phi for the first level. Phi shape: (37739, 5). First words: MultiIndex([('@lemmatized',     'hazardous'),
            ('@lemmatized', 'expostulation'),
            ('@lemmatized',        'hearse'),
            ('@lemmatized',      'valkyrie'),
            ('@lemmatized',     'sagebrush'),
            ('@lemmatized',        'oldham'),
            ('@lemmatized',   'indefinable'),
            ('@lemmatized',       'bellboy'),
 

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 23878.166015625, 'coherence_20': 1.229698071524603, 'diversity_euclidean': 0.0582555565105075, 'diversity_jensenshannon': 0.6403552575649625, 'diversity_hellinger': 0.74680146288752, 'diversity_cosine': 0.8519951977197273, 'perplexity': 23878.166015625, 'ppl_fair': 23878.166015625, 'ppl_cheatty': 6565.55517578125}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.22it/s]
Creating first level with 5 topics. Dictionary: artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739).
Copying phi for the first level. Phi shape: (37739, 5). First words: MultiIndex([('@lemmatized',     'hazardous'),
            ('@lemmatized', 'expostulation'),
            ('@lemmatized',        'hearse'),
            ('@lemmatized',      'valkyrie'),
            ('@lemmatized',     'sagebrush'),
            ('@lemmatized',        'oldham'),
            ('@lemmatized',   'indefinable'),
            ('@lemmatized',       'bellboy'),
 

...

Bank scores: {'perplexity_score': 19128.2890625, 'coherence_20': 1.151999215668335, 'diversity_euclidean': 0.0675270493926133, 'diversity_jensenshannon': 0.6725448126230565, 'diversity_hellinger': 0.7855662025219016, 'diversity_cosine': 0.8670391618163544, 'perplexity': 19128.2890625, 'ppl_fair': 19128.2890625, 'ppl_cheatty': 6197.19140625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.30it/s]
Creating first level with 7 topics. Dictionary: artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739).
Copying phi for the first level. Phi shape: (37739, 7). First words: MultiIndex([('@lemmatized',     'hazardous'),
            ('@lemmatized', 'expostulation'),
            ('@lemmatized',        'hearse'),
            ('@lemmatized',      'valkyrie'),
            ('@lemmatized',     'sagebrush'),
            ('@lemmatized',        'oldham'),
            ('@lemmatized',   'indefinable'),
            ('@lemmatized',       'bellboy'),
        

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 19128.2890625, 'coherence_20': 1.151999215668335, 'diversity_euclidean': 0.0675270493926133, 'diversity_jensenshannon': 0.6725448126230565, 'diversity_hellinger': 0.7855662025219016, 'diversity_cosine': 0.8670391618163544, 'perplexity': 19128.2890625, 'ppl_fair': 19128.2890625, 'ppl_cheatty': 6197.19140625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.54it/s]
Creating first level with 7 topics. Dictionary: artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739).
Copying phi for the first level. Phi shape: (37739, 7). First words: MultiIndex([('@lemmatized',     'hazardous'),
            ('@lemmatized', 'expostulation'),
            ('@lemmatized',        'hearse'),
            ('@lemmatized',      'valkyrie'),
            ('@lemmatized',     'sagebrush'),
            ('@lemmatized',        'oldham'),
            ('@lemmatized',   'indefinable'),
            ('@lemmatized',       'bellboy'),
        

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 17225.158203125, 'coherence_20': 1.1539726713344447, 'diversity_euclidean': 0.06532281912944632, 'diversity_jensenshannon': 0.6669165021161831, 'diversity_hellinger': 0.7783840409048797, 'diversity_cosine': 0.8626256927229845, 'perplexity': 17225.158203125, 'ppl_fair': 17225.158203125, 'ppl_cheatty': 6089.13037109375}
100%|███████████████████████████████████████████████| 20/20 [27:07<00:00, 81.39s/it]
CPU times: user 31min 11s, sys: 56.2 s, total: 32min 7s
Wall time: 27min 7s


What topics we have in bank

In [53]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0   
            expostulation      0.0      0.0      0.0      0.0      0.0   
            hearse             0.0      0.0      0.0      0.0      0.0   
            valkyrie           0.0      0.0      0.0      0.0      0.0   
            sagebrush          0.0      0.0      0.0      0.0      0.0   

                           topic_5  topic_6  topic_7  
@lemmatized hazardous          0.0      0.0      0.0  
            expostulation      0.0      0.0      0.0  
            hearse             0.0      0.0      0.0  
            valkyrie           0.0      0.0      0.0  
            sagebrush          0.0      0.0      0.0

In [54]:
bank_topics = optimizer._topic_bank.view_topics()

In [55]:
bank_topics.shape

(37739, 8)

In [56]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0   
            expostulation      0.0      0.0      0.0      0.0      0.0   
            hearse             0.0      0.0      0.0      0.0      0.0   
            valkyrie           0.0      0.0      0.0      0.0      0.0   
            sagebrush          0.0      0.0      0.0      0.0      0.0   

                           topic_5  topic_6  topic_7  
@lemmatized hazardous          0.0      0.0      0.0  
            expostulation      0.0      0.0      0.0  
            hearse             0.0      0.0      0.0  
            valkyrie           0.0      0.0      0.0  
            sagebrush          0.0      0.0      0.0

In [57]:
bank_topics['topic_0'].sort_values(ascending=False)[:20]

@lemmatized  constantine    0.017840
             roman          0.013013
             emperor        0.012049
             empire         0.008822
             augustus       0.007161
             diocletian     0.006886
             p              0.006624
             rome           0.006320
             barnes         0.006038
             julian         0.006024
             eusebius       0.005116
             reign          0.004948
             domitian       0.004876
             constantius    0.004788
             christian      0.004071
             army           0.003807
             hadrian        0.003765
             justinian      0.003478
             city           0.003428
             military       0.003293
Name: topic_0, dtype: float64

And topic scores

In [58]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7
kernel_size,4118.000000,4699.000000,4803.000000,5443.000000,5565.000000,2541.000000,4029.000000,4762.000000
coherence_20,1.131275,1.083500,1.043963,1.088446,1.801306,1.017304,0.898200,1.167787
distance_to_nearest,0.000000,0.874184,0.788165,0.841812,0.710506,0.866538,0.877169,0.805760


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [59]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [60]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [61]:
optimizer._result['num_bank_topics']

[5, 5, 5, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8]

In [62]:
len(optimizer._result['bank_topic_scores'])

20

In [63]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [64]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [65]:
NUM_ITERATIONS = 20

In [66]:
seed = 0

In [67]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [68]:
DATASET_INTERNALS_FOLDER_PATH

'./RTL_Wiki_Person__internals'

In [69]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2_50'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [70]:
SEARCH_RESULTS_FOLDER_PATH

'./RTL_Wiki_Person__internals/result2_50'

In [71]:
BANK_FOLDER_PATH

'./RTL_Wiki_Person__internals/result2_50/bank__0'

In [72]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [73]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [74]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 0.7938587462014927,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:208: UserWarning: topic_score_threshold_percentile 0.7938587462014927 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(


In [75]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [76]:
! ls ./Post_Science__internals

batches    _result  _result2  result2_50  result_unfiltered_dict
dict.dict  result   result2   result_50   vw.txt


In [77]:
optimizer._save_file_path

'./RTL_Wiki_Person__internals/result2_50/search_result__0.json'

In [78]:
optimizer._topic_bank._path

'./RTL_Wiki_Person__internals/result2_50/bank__0'

Fulfilling the search (get ready for a really long process!):

In [79]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                        | 0/20 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 50 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.85it/s]
Using absoulte threshold: 0.7938587462014927.
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 16077.958984375, 'coherence_20': 1.1225033344850237, 'diversity_euclidean': 0.06877371410525385, 'diversity_jensenshannon': 0.687365105375971, 'diversity_hellinger': 0.804517438601834, 'diversity_cosine': 0.8903542350272428, 'perplexity': 16077.958984375, 'ppl_fair': 16077.958984375, 'ppl_cheatty': 5728.96484375}
  5%|██▎                                            | 1/20 [01:42<32:29, 102.60s/it]No spec topics
No spec topics


...

Bank scores: {'perplexity_score': 9492.37109375, 'coherence_20': 1.0967409799884282, 'diversity_euclidean': 0.06884468919017744, 'diversity_jensenshannon': 0.6858695825538061, 'diversity_hellinger': 0.8040435784546612, 'diversity_cosine': 0.8783290812834309, 'perplexity': 9492.37109375, 'ppl_fair': 9492.37109375, 'ppl_cheatty': 5078.76904296875}
 90%|█████████████████████████████████████████▍    | 18/20 [33:18<03:53, 116.83s/it]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 50 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.02it/s]
Using absoulte threshold: 0.7938587462014927.
Creating first level with 21 topics. Dictionary: artm.Dictionary(name=68b4bcd8-474f-49ce-9b78-de418b7c1481, num_entries=37739).
Copying phi for the first level. Phi shape: (37739, 21). First words: MultiIndex([('@lemmatized',     'hazardous'),
            ('@lemmatized', 'expostulation'),
            ('@lemmatized',        'hearse'),
            ('@lemmatized',      'valkyrie'),
          

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 9192.8720703125, 'coherence_20': 1.0773131766905026, 'diversity_euclidean': 0.06746011408265991, 'diversity_jensenshannon': 0.6825363424503622, 'diversity_hellinger': 0.7998098519245985, 'diversity_cosine': 0.875497499598681, 'perplexity': 9192.8720703125, 'ppl_fair': 9192.8720703125, 'ppl_cheatty': 5054.6123046875}
100%|██████████████████████████████████████████████| 20/20 [37:03<00:00, 111.16s/it]
CPU times: user 42min 39s, sys: 1min 14s, total: 43min 54s
Wall time: 37min 3s


In [82]:
optimizer._main_modality

'@lemmatized'

What topics we have in bank

In [83]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0   
            expostulation      0.0      0.0      0.0      0.0      0.0   
            hearse             0.0      0.0      0.0      0.0      0.0   
            valkyrie           0.0      0.0      0.0      0.0      0.0   
            sagebrush          0.0      0.0      0.0      0.0      0.0   

                           topic_5  topic_6  topic_7  topic_8   topic_9  ...  \
@lemmatized hazardous          0.0      0.0      0.0      0.0  0.000000  ...   
            expostulation      0.0      0.0      0.0      0.0  0.000000  ...   
            hearse             0.0      0.0      0.0      0.0  0.000019  ...   
            valkyrie           0.0      0.0      0.0      0.0  0.000000  ...   
            sagebrush          0.0      0.0      0.0      0.0  0.000000  ...   

                           topic_11  topic_12  topic_13  topic_14  topic_15  \
@lemmatized hazardous           0.0       0.0       0.0       0.0       0.0   
            expostulation       0.0       0.0       0.0       0.0       0.0   
            hearse              0.0       0.0       0.0       0.0       0.0   
            valkyrie            0.0       0.0       0.0       0.0       0.0   
            sagebrush           0.0       0.0       0.0       0.0       0.0   

                           topic_16  topic_17  topic_18  topic_19  topic_20  
@lemmatized hazardous           0.0       0.0  0.000000       0.0       0.0  
            expostulation       0.0       0.0  0.000044       0.0       0.0  
            hearse              0.0       0.0  0.000000       0.0       0.0  
            valkyrie            0.0       0.0  0.000000       0.0       0.0  
            sagebrush           0.0       0.0  0.000000       0.0       0.0  

[5 rows x 21 columns]

In [84]:
bank_topics = optimizer._topic_bank.view_topics()

In [85]:
bank_topics.shape

(37739, 21)

In [86]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0   
            expostulation      0.0      0.0      0.0      0.0      0.0   
            hearse             0.0      0.0      0.0      0.0      0.0   
            valkyrie           0.0      0.0      0.0      0.0      0.0   
            sagebrush          0.0      0.0      0.0      0.0      0.0   

                           topic_5  topic_6  topic_7  topic_8   topic_9  ...  \
@lemmatized hazardous          0.0      0.0      0.0      0.0  0.000000  ...   
            expostulation      0.0      0.0      0.0      0.0  0.000000  ...   
            hearse             0.0      0.0      0.0      0.0  0.000019  ...   
            valkyrie           0.0      0.0      0.0      0.0  0.000000  ...   
            sagebrush          0.0      0.0      0.0      0.0  0.000000  ...   

                           topic_11  topic_12  topic_13  topic_14  topic_15  \
@lemmatized hazardous           0.0       0.0       0.0       0.0       0.0   
            expostulation       0.0       0.0       0.0       0.0       0.0   
            hearse              0.0       0.0       0.0       0.0       0.0   
            valkyrie            0.0       0.0       0.0       0.0       0.0   
            sagebrush           0.0       0.0       0.0       0.0       0.0   

                           topic_16  topic_17  topic_18  topic_19  topic_20  
@lemmatized hazardous           0.0       0.0  0.000000       0.0       0.0  
            expostulation       0.0       0.0  0.000044       0.0       0.0  
            hearse              0.0       0.0  0.000000       0.0       0.0  
            valkyrie            0.0       0.0  0.000000       0.0       0.0  
            sagebrush           0.0       0.0  0.000000       0.0       0.0  

[5 rows x 21 columns]

In [87]:
bank_topics['topic_7'].sort_values(ascending=False)[:20]

@lemmatized  emperor      0.055042
             imperial     0.032100
             japan        0.024586
             prince       0.023391
             daughter     0.016073
             japanese     0.015144
             princess     0.014929
             fujiwara     0.014755
             p            0.013037
             reign        0.012029
             donaldson    0.009103
             court        0.008996
             empress      0.008289
             brown        0.008168
             minamoto     0.008114
             varley       0.007744
             throne       0.007293
             titsingh     0.007006
             pp           0.006618
             month        0.005856
Name: topic_7, dtype: float64

And topic scores

In [88]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,...,topic_11,topic_12,topic_13,topic_14,topic_15,topic_16,topic_17,topic_18,topic_19,topic_20
kernel_size,3669.000000,3826.000000,3891.000000,3566.000000,3153.000000,4026.000000,3564.000000,1701.000000,4146.000000,3762.000000,...,2869.000000,3096.000000,3406.000000,3213.000000,3485.000000,2956.000000,3598.000000,4309.000000,3316.000000,4387.000000
coherence_20,1.149359,1.094059,1.510019,1.042494,0.806954,1.897256,0.971696,1.017304,0.869236,0.925371,...,1.592502,0.934561,1.191814,1.079696,0.898798,1.046236,0.834385,0.825890,0.860103,0.947832
distance_to_nearest,0.000000,0.829735,0.843415,0.816017,0.810174,0.762359,0.838471,0.896446,0.818576,0.795369,...,0.516390,0.882426,0.582541,0.521878,0.791107,0.838275,0.619696,0.837643,0.805003,0.691188


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [89]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [90]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [91]:
optimizer._result['num_bank_topics']

[11,
 11,
 11,
 12,
 13,
 14,
 14,
 14,
 14,
 15,
 16,
 16,
 16,
 16,
 19,
 20,
 20,
 21,
 21,
 21]

In [92]:
len(optimizer._result['bank_topic_scores'])

20

In [93]:
optimizer._result

{'optimum': 21,
 'optimum_std': 0.0,
 'bank_scores': [{'perplexity_score': 16077.958984375,
   'coherence_20': 1.1225033344850237,
   'diversity_euclidean': 0.06877371410525385,
   'diversity_jensenshannon': 0.687365105375971,
   'diversity_hellinger': 0.804517438601834,
   'diversity_cosine': 0.8903542350272428,
   'perplexity': 16077.958984375,
   'ppl_fair': 16077.958984375,
   'ppl_cheatty': 5728.96484375},
  {'perplexity_score': 16077.958984375,
   'coherence_20': 1.1225033344850237,
   'diversity_euclidean': 0.06877371410525385,
   'diversity_jensenshannon': 0.687365105375971,
   'diversity_hellinger': 0.804517438601834,
   'diversity_cosine': 0.8903542350272428,
   'perplexity': 16077.958984375,
   'ppl_fair': 16077.958984375,
   'ppl_cheatty': 5728.96484375},
  {'perplexity_score': 16077.958984375,
   'coherence_20': 1.1225033344850237,
   'diversity_euclidean': 0.06877371410525385,
   'diversity_jensenshannon': 0.687365105375971,
   'diversity_hellinger': 0.804517438601834,
  

In [94]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 3669,
   'coherence_20': 1.149358735569971,
   'distance_to_nearest': 0.0},
  {'kernel_size': 3826,
   'coherence_20': 1.0940593226519857,
   'distance_to_nearest': 0.8297350277944032},
  {'kernel_size': 2306,
   'coherence_20': 1.1740822671595732,
   'distance_to_nearest': 0.9141673378661013},
  {'kernel_size': 3490,
   'coherence_20': 1.0702254526038417,
   'distance_to_nearest': 0.9036188885034283},
  {'kernel_size': 3277,
   'coherence_20': 0.8372213061875133,
   'distance_to_nearest': 0.8369490437335917},
  {'kernel_size': 3891,
   'coherence_20': 1.510018973038721,
   'distance_to_nearest': 0.8434147923466643},
  {'kernel_size': 2925,
   'coherence_20': 0.7941706697558051,
   'distance_to_nearest': 0.8281433389755726},
  {'kernel_size': 3566,
   'coherence_20': 1.0424944557666154,
   'distance_to_nearest': 0.8160167830892179},
  {'kernel_size': 3153,
   'coherence_20': 0.8069537544283898,
   'distance_to_nearest': 0.8101736562618436},
  {'kernel_size': 4026,
   

In [95]:
optimizer._result['model_scores'][0]

{'perplexity_score': 3041.67138671875,
 'coherence_20': 0.5898700966827493,
 'diversity_euclidean': 0.062318714185795654,
 'diversity_jensenshannon': 0.6669767696232136,
 'diversity_hellinger': 0.7785279346901158,
 'diversity_cosine': 0.8602338865058863,
 'perplexity': 3041.67138671875}

In [126]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 10

0.9058404227165608

In [127]:
len(optimizer._result['bank_scores'])

20

In [128]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 13329.8876953125,
 'coherence_20': 0.9058404227165606,
 'diversity_euclidean': 0.05184720024244504,
 'diversity_jensenshannon': 0.6500697527850625,
 'diversity_hellinger': 0.7586476679507803,
 'diversity_cosine': 0.8126390939253648,
 'perplexity': 13329.8876953125,
 'ppl_fair': 13329.8876953125,
 'ppl_cheatty': 5741.56787109375}

In [129]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 3810, 'coherence_20': 0.5225908625149936},
  {'kernel_size': 3895, 'coherence_20': 0.5331450869041109},
  {'kernel_size': 3987,
   'coherence_20': 1.284041737255726,
   'distance_to_nearest': 0.0},
  {'kernel_size': 3896, 'coherence_20': 0.5261857613384362},
  {'kernel_size': 3860, 'coherence_20': 0.3796521580255667},
  {'kernel_size': 3654, 'coherence_20': 0.5656146718404065},
  {'kernel_size': 3844, 'coherence_20': 0.44309547614289096},
  {'kernel_size': 3919, 'coherence_20': 0.5484668820539618},
  {'kernel_size': 4084, 'coherence_20': 0.5739931717770947},
  {'kernel_size': 2794, 'coherence_20': 0.5111064072825383},
  {'kernel_size': 4275, 'coherence_20': 0.5617371534777974},
  {'kernel_size': 3470, 'coherence_20': 0.6047710619718488},
  {'kernel_size': 3392, 'coherence_20': 0.6510584572257663},
  {'kernel_size': 3715,
   'coherence_20': 0.9103697786572175,
   'distance_to_nearest': 0.8389113266096888},
  {'kernel_size': 3735,
   'coherence_20': 0.7229039920008129,


In [130]:
! echo $SEARCH_RESULTS_FOLDER_PATH
! ls $SEARCH_RESULTS_FOLDER_PATH

./RTL_Wiki_Person__internals/result2
bank__0  search_result__0.json


In [131]:
! ls Good_RU_Wiki__internals

batches  dict.dict  result  result2  vw.txt


In [96]:
optimizer._save_file_path

'./RTL_Wiki_Person__internals/result2_50/search_result__0.json'